# 🎲 LAB-04 Playground: Inference Math, Sampling & Quantization

Welcome to the **LAB-04 Interactive Playground**! In this notebook, you will interactively experiment with:
1. **Logit Transformations & Token Sampling:** Temperature Scaling, Top-K, Top-P (Nucleus), and Min-P filtering.
2. **Post-Training Quantization (FP32 -> INT8 / INT4):** Symmetric and Asymmetric linear quantization, scale/zero-point derivation, and SQNR signal preservation metrics.
3. **Inference Economics Calculator:** Dynamic KV Cache memory allocation, parameter VRAM overhead, and FLOP throughput bounds.

In [ ]:
# Setup environment path
import sys
from pathlib import Path

# Locate repository root dynamically
current_dir = Path.cwd()
root_dir = current_dir
while root_dir.parent != root_dir and not (root_dir / "pyproject.toml").exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import math
import torch
import torch.nn.functional as F

from projects.a_llm_basics.src.lab_04_inference import InferenceCostCalculator, SamplingEngine, UniformQuantizer, compute_cross_entropy_loss, compute_perplexity, compute_quantization_error, compute_shannon_entropy

print("✅ LAB-04 Modules imported successfully!")

---
## 1. 🌡️ Sampling Strategies: Temperature, Top-K & Top-P

In [ ]:
# Simulated unnormalized logits vector for a 5-token vocabulary
logits = torch.tensor([[12.0, 8.0, 4.0, 1.0, 0.2]])
vocab = ["llama", "transformer", "attention", "token", "cat"]

print("Raw Logits:", logits.numpy()[0])

# Experiment 1: Temperature Scaling
for temp in [0.1, 0.7, 1.0, 2.0]:
    scaled = logits / temp
    probs = F.softmax(scaled, dim=-1)[0]
    entropy = compute_shannon_entropy(probs).item()
    print(f"T={temp:<4.1f} | Probs: {probs.numpy().round(3)} | Entropy: {entropy:.2f} bits")

---
## 2. ⚡ Quantization & Precision Loss Metrics (FP32 -> INT8)

In [ ]:
torch.manual_seed(42)
weights_fp32 = torch.randn(1000)

# Symmetric INT8 Quantization
quantizer_sym = UniformQuantizer(num_bits=8, symmetric=True)
q_sym, scale_sym, zero_sym = quantizer_sym.quantize(weights_fp32)
deq_sym = quantizer_sym.dequantize(q_sym, scale_sym, zero_sym)
metrics_sym = compute_quantization_error(weights_fp32, deq_sym)

print("=== Symmetric INT8 Quantization Metrics ===")
print(f"Scale Factor (S): {scale_sym:.6f}")
print(f"MSE: {metrics_sym['mse']:.6f}")
print(f"Cosine Similarity: {metrics_sym['cosine_similarity']:.6f}")
print(f"SQNR: {metrics_sym['sqnr_db']:.2f} dB")

---
## 3. 🖥️ Inference Economics: VRAM & KV Cache Calculator

In [ ]:
# Llama-3-8B architecture specifications
calc = InferenceCostCalculator(
    num_layers=32,
    num_heads=32,
    head_dim=128,
    hidden_size=4096,
    num_kv_heads=8  # Grouped-Query Attention (GQA 4:1 ratio)
)

num_params = 8_000_000_000
weight_vram_gb = calc.compute_model_param_bytes(num_params, precision_bytes=2.0) / (1024**3)
kv_vram_mb = calc.compute_kv_cache_bytes(batch_size=1, seq_len=8192, precision_bytes=2.0) / (1024**2)
max_batch = calc.compute_max_batch_size(vram_capacity_bytes=24 * (1024**3), num_params=num_params, seq_len=8192)

print("=== Llama-3-8B Inference Profile (FP16) ===")
print(f"Weights VRAM: {weight_vram_gb:.2f} GB")
print(f"KV Cache (B=1, S=8192): {kv_vram_mb:.2f} MB")
print(f"Max Batch Size on RTX 4090 (24GB VRAM): {max_batch} sequences")